# Panel Regression: Fixed & Random Effects

## Research Question
**Is biodiversity loss in beef-exporting countries associated with the volume of beef exports to the United Kingdom?**

## Approach
We use the **historical panel (2001–2014)** created in the data-cleaning notebook
(historical BIO 2001–2014 merged with FAOSTAT bilateral trade 2001–2014).

| Model | What it controls for |
|-------|---------------------|
| **Pooled OLS** | Baseline - ignores panel structure |
| **Fixed Effects (FE)** | All *time-invariant* country characteristics |
| **Random Effects (RE)** | Treats country effects as random draws - more efficient if valid |
| **Hausman test** | Decides whether FE or RE is appropriate |

### Model Specification
$$\text{log\_uk\_import\_qty\_t}_{it} = \alpha_i + \beta \cdot \text{bii\_loss}_{it} + \varepsilon_{it}$$

- $i$ = country, $t$ = year
- $\alpha_i$ = country-specific intercept (fixed or random)
- All regressions use **clustered standard errors** at the country level


---
## 0 - Setup


In [29]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from linearmodels.panel import compare

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)
sns.set_style('whitegrid')

# Paths (notebook lives in data/)
DATA  = os.path.dirname(os.path.abspath('__file__'))
CLEAN = os.path.join(DATA, 'clean')

print('Ready.')


Ready.


---
## 1 - Load Historical Panel (2001–2014)


In [30]:
panel = pd.read_csv(os.path.join(CLEAN, 'panel_2001_2014.csv'))

print(f'Shape: {panel.shape}')
print(f'Countries: {panel["iso3"].nunique()}')
print(f'Years: {sorted(panel["year"].unique())}')
panel.head()


Shape: (340, 7)
Countries: 37
Years: [np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014)]


,iso3,country,year,uk_import_qty_t,log_uk_import_qty_t,bii,bii_loss
0,ARE,United Arab Emirates,2005,16.0,2.833213,1.000000,0.000000
1,ARG,Argentina,2001,5652.0,8.639942,0.761125,0.238875
2,ARG,Argentina,2002,6247.0,8.740017,0.759830,0.240170
3,ARG,Argentina,2003,6065.0,8.710455,0.757173,0.242827
4,ARG,Argentina,2004,9796.0,9.189831,0.752349,0.247651


### 1.1 - Panel Balance Check
The panel is **unbalanced** - not every country is observed in every year.
Both `PanelOLS` (FE) and `RandomEffects` (RE) handle unbalanced panels correctly.


In [31]:
balance = panel.groupby('iso3')['year'].count().describe()
print('Observations per country (summary):')
print(balance.round(1))

presence = panel.pivot_table(index='iso3', columns='year',
                             values='bii_loss', aggfunc='count')
print(f'\nCountries with all 14 years: {(presence.sum(axis=1) == 14).sum()}')
print(f'Countries with < 5 years:    {(presence.sum(axis=1) < 5).sum()}')


Observations per country (summary):
count    37.0
mean      9.2
std       4.9
min       1.0
25%       5.0
50%      11.0
75%      14.0
max      14.0
Name: year, dtype: float64

Countries with all 14 years: 12
Countries with < 5 years:    8


### 1.2 - Descriptive Statistics


In [32]:
desc_cols = ['bii', 'bii_loss', 'uk_import_qty_t', 'log_uk_import_qty_t']
print('Overall summary statistics:')
panel[desc_cols].describe().round(4)


Overall summary statistics:


,bii,bii_loss,uk_import_qty_t,log_uk_import_qty_t
count,340.0000,340.0000,340.0000,340.0000
mean,0.6752,0.3248,4344.7469,4.6977
std,0.1432,0.1432,12437.9420,3.1170
min,0.3323,0.0000,0.0100,0.0100
25%,0.6078,0.2425,4.0000,1.6094
50%,0.6700,0.3300,108.0000,4.6913
75%,0.7575,0.3922,1167.3500,7.0633
max,1.0000,0.6677,60191.0000,11.0053


### 1.3 - Within vs. Between Variation
Fixed effects exploit **within-country** variation over time.
If most variation in `bii_loss` is *between* countries, FE will have low power.


In [33]:
for col in ['bii_loss', 'log_uk_import_qty_t']:
    overall_var = panel[col].var()
    between_var = panel.groupby('iso3')[col].mean().var()
    within_var  = panel.groupby('iso3')[col].apply(lambda x: x - x.mean()).var()
    print(f'{col}:')
    print(f'  Overall variance:  {overall_var:.6f}')
    print(f'  Between variance:  {between_var:.6f}  ({between_var/overall_var*100:.1f}%)')
    print(f'  Within variance:   {within_var:.6f}  ({within_var/overall_var*100:.1f}%)')
    print()


bii_loss:
  Overall variance:  0.020515
  Between variance:  0.021234  (103.5%)
  Within variance:   0.000096  (0.5%)

log_uk_import_qty_t:
  Overall variance:  9.715977
  Between variance:  8.407175  (86.5%)
  Within variance:   0.902954  (9.3%)



---
## 2 - Set Multi-Index for Panel Models
`linearmodels` requires a `MultiIndex` of `(entity, time)` - here `(iso3, year)`.


In [34]:
panel = panel.set_index(['iso3', 'year'])
panel.index.names = ['iso3', 'year']
print(f'Index: {panel.index.names}')
panel.head()


Index: ['iso3', 'year']


country  uk_import_qty_t  log_uk_import_qty_t       bii  bii_loss
iso3 year                                                                                
ARE  2005  United Arab Emirates             16.0             2.833213  1.000000  0.000000
ARG  2001             Argentina           5652.0             8.639942  0.761125  0.238875
     2002             Argentina           6247.0             8.740017  0.759830  0.240170
     2003             Argentina           6065.0             8.710455  0.757173  0.242827
     2004             Argentina           9796.0             9.189831  0.752349  0.247651

---
## 3 - Pooled OLS (Baseline)
Ignores the panel structure - treats all observations as independent.
Standard errors are **clustered at the country level**.


In [35]:
dep_var = panel['log_uk_import_qty_t']
exog    = sm.add_constant(panel[['bii_loss']])

pooled_model = PooledOLS(dep_var, exog)
pooled_res   = pooled_model.fit(cov_type='clustered', cluster_entity=True)

print(pooled_res.summary)


                           PooledOLS Estimation Summary                          
Dep. Variable:     log_uk_import_qty_t   R-squared:                        0.1300
Estimator:                   PooledOLS   R-squared (Between):              0.0651
No. Observations:                  340   R-squared (Within):              -0.0435
Date:                 Thu, Apr 02 2026   R-squared (Overall):              0.1300
Time:                         21:01:48   Log-likelihood                   -844.80
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      50.522
Entities:                           37   P-value                           0.0000
Avg Obs:                        9.1892   Distribution:                   F(1,338)
Min Obs:                        1.0000                                           
Max Obs:                        14.000   F-statistic (robust):             5.4623
                

---
## 4 - Fixed Effects (FE) - Entity Demeaning

Controls for all **time-invariant** country characteristics.
The coefficient on `bii_loss` captures the **within-country** relationship:
*when a country's biodiversity loss increases over time, do its beef exports to the UK change?*


In [36]:
# 4a. Entity FE only
fe_model = PanelOLS(dep_var, panel[['bii_loss']], entity_effects=True)
fe_res   = fe_model.fit(cov_type='clustered', cluster_entity=True)

print('=' * 70)
print('FIXED EFFECTS - Entity only')
print('=' * 70)
print(fe_res.summary)


FIXED EFFECTS - Entity only
                           PanelOLS Estimation Summary                           
Dep. Variable:     log_uk_import_qty_t   R-squared:                        0.0521
Estimator:                    PanelOLS   R-squared (Between):             -4.8411
No. Observations:                  340   R-squared (Within):               0.0521
Date:                 Thu, Apr 02 2026   R-squared (Overall):             -4.2997
Time:                         21:01:49   Log-likelihood                   -455.48
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      16.611
Entities:                           37   P-value                           0.0001
Avg Obs:                        9.1892   Distribution:                   F(1,302)
Min Obs:                        1.0000                                           
Max Obs:                        14.000   F-statistic (robust):        

In [37]:
# 4b. Entity + Time FE (two-way)
fe2_model = PanelOLS(dep_var, panel[['bii_loss']],
                     entity_effects=True, time_effects=True)
fe2_res   = fe2_model.fit(cov_type='clustered', cluster_entity=True)

print('=' * 70)
print('FIXED EFFECTS - Entity + Time (two-way)')
print('=' * 70)
print(fe2_res.summary)


FIXED EFFECTS - Entity + Time (two-way)
                           PanelOLS Estimation Summary                           
Dep. Variable:     log_uk_import_qty_t   R-squared:                        0.0492
Estimator:                    PanelOLS   R-squared (Between):             -4.5797
No. Observations:                  340   R-squared (Within):               0.0521
Date:                 Thu, Apr 02 2026   R-squared (Overall):             -4.0727
Time:                         21:01:49   Log-likelihood                   -446.71
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      14.964
Entities:                           37   P-value                           0.0001
Avg Obs:                        9.1892   Distribution:                   F(1,289)
Min Obs:                        1.0000                                           
Max Obs:                        14.000   F-statistic (robu

### 4.1 - F-test for Joint Significance of Entity Effects
Tests whether the country fixed effects are *jointly significant*.


In [38]:
print('F-test for poolability (entity effects = 0):')
print(f'  F-statistic: {fe_res.f_poolable.stat:.4f}')
print(f'  p-value:     {fe_res.f_poolable.pval:.6f}')
if fe_res.f_poolable.pval < 0.05:
    print('  => Reject H0: entity effects ARE significant - pooled OLS is inappropriate.')
else:
    print('  => Cannot reject H0: entity effects not significant - pooled OLS may suffice.')


F-test for poolability (entity effects = 0):


AttributeError: 'PanelEffectsResults' object has no attribute 'f_poolable'

---
## 5 - Random Effects (RE)

RE is more **efficient** than FE (uses both within *and* between variation)
but requires that country effects are uncorrelated with the regressors.
The Hausman test (next section) evaluates this assumption.


In [ ]:
re_model = RandomEffects(dep_var, exog)
re_res   = re_model.fit(cov_type='clustered', cluster_entity=True)

print('=' * 70)
print('RANDOM EFFECTS')
print('=' * 70)
print(re_res.summary)


---
## 6 - Hausman Test: FE vs RE

- **H0**: RE is consistent and efficient (country effects uncorrelated with regressors).
- **H1**: RE is inconsistent - use FE.

We compare the `bii_loss` coefficient from both models.


In [ ]:
# Manual Hausman test
b_fe = fe_res.params['bii_loss']
b_re = re_res.params['bii_loss']

var_fe = fe_res.std_errors['bii_loss'] ** 2
var_re = re_res.std_errors['bii_loss'] ** 2

var_diff = var_fe - var_re

if var_diff > 0:
    hausman_stat = (b_fe - b_re) ** 2 / var_diff
    hausman_pval = 1 - stats.chi2.cdf(hausman_stat, df=1)

    print('Hausman Test  (FE vs RE)')
    print('=' * 50)
    print(f'  FE coeff (bii_loss):  {b_fe:>10.4f}')
    print(f'  RE coeff (bii_loss):  {b_re:>10.4f}')
    print(f'  Difference:           {b_fe - b_re:>10.4f}')
    print(f'  Hausman chi2(1):      {hausman_stat:>10.4f}')
    print(f'  p-value:              {hausman_pval:>10.6f}')
    print()
    if hausman_pval < 0.05:
        print('  => Reject H0 at 5%: FE is preferred (RE is inconsistent).')
        preferred = 'FE'
    else:
        print('  => Cannot reject H0: RE is preferred (more efficient & consistent).')
        preferred = 'RE'
else:
    print('  Var(b_FE) < Var(b_RE) -- Hausman statistic is negative.')
    print('  This can happen in finite samples with clustered SEs.')
    print('  Interpreting as: no strong evidence against RE.')
    preferred = 'RE'

print(f'\n  >> Preferred model: {preferred}')


---
## 7 - Model Comparison Table


In [ ]:
comparison = compare(
    {
        'Pooled OLS':     pooled_res,
        'FE (entity)':    fe_res,
        'FE (two-way)':   fe2_res,
        'Random Effects': re_res,
    },
    stars=True,
)
print(comparison.summary)


---
## 8 - Diagnostics


### 8.1 - Residuals from the Preferred Model


In [ ]:
preferred_res = fe_res if preferred == 'FE' else re_res
resids = preferred_res.resids.copy()
fitted = preferred_res.fitted_values.copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) Residuals vs Fitted
axes[0].scatter(fitted, resids, alpha=0.5, s=20)
axes[0].axhline(0, color='red', linestyle='--', linewidth=0.8)
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title(f'Residuals vs Fitted ({preferred})')

# (b) Histogram of residuals
axes[1].hist(resids, bins=25, edgecolor='white', alpha=0.7)
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

# (c) Q-Q plot
sm.qqplot(resids.values.flatten(), line='45', ax=axes[2], alpha=0.5)
axes[2].set_title('Q-Q Plot (Normal)')

plt.tight_layout()
plt.savefig(os.path.join(CLEAN, 'fe_re_diagnostics.png'), dpi=150,
            bbox_inches='tight')
plt.show()
print('Saved => data/clean/fe_re_diagnostics.png')


### 8.2 - Country Fixed Effects (Estimated alpha_i)


In [ ]:
if preferred == 'FE':
    effects = fe_res.estimated_effects.copy()
    effects.columns = ['alpha_i']
    effects = (effects.reset_index()
              .drop_duplicates(subset='iso3')
              .set_index('iso3')
              .sort_values('alpha_i', ascending=True))

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(effects.index, effects['alpha_i'],
            color='steelblue', edgecolor='white')
    ax.set_xlabel('Estimated country fixed effect (alpha_i)')
    ax.set_title('Country Fixed Effects - log(UK beef import qty)')
    ax.axvline(0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(CLEAN, 'country_fixed_effects.png'), dpi=150,
                bbox_inches='tight')
    plt.show()
    print('Saved => data/clean/country_fixed_effects.png')
else:
    print('RE preferred - no entity-specific intercepts to plot.')


### 8.3 - Within-Country Time Trends (Spaghetti Plot)


In [ ]:
plot_df = panel.reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for iso3, grp in plot_df.groupby('iso3'):
    axes[0].plot(grp['year'], grp['bii_loss'],
                marker='.', alpha=0.4, linewidth=0.8)
    axes[1].plot(grp['year'], grp['log_uk_import_qty_t'],
                marker='.', alpha=0.4, linewidth=0.8)

axes[0].set_title('BII Loss by Country over Time')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('bii_loss')

axes[1].set_title('Log UK Beef Import Qty by Country over Time')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('log(1 + qty)')

plt.tight_layout()
plt.savefig(os.path.join(CLEAN, 'spaghetti_trends.png'), dpi=150,
            bbox_inches='tight')
plt.show()
print('Saved => data/clean/spaghetti_trends.png')


---
## 9 - Robustness Checks


### 9.1 - Restrict to 2010-2014 Sub-Period
Tests whether results hold on the shorter 2010-2014 window alone.


In [ ]:
panel_short = panel[panel.reset_index()['year'].values >= 2010].copy()
print(f'2010-2014 sub-panel: {panel_short.shape[0]} obs, '
      f'{panel_short.reset_index()["iso3"].nunique()} countries, '
      f'{panel_short.reset_index()["year"].nunique()} years')

dep_short = panel_short['log_uk_import_qty_t']

# FE on 2010-2014 only
fe_short = PanelOLS(dep_short, panel_short[['bii_loss']], entity_effects=True)
fe_short_res = fe_short.fit(cov_type='clustered', cluster_entity=True)

# RE on 2010-2014 only
exog_short = sm.add_constant(panel_short[['bii_loss']])
re_short = RandomEffects(dep_short, exog_short)
re_short_res = re_short.fit(cov_type='clustered', cluster_entity=True)

print('\n--- FE (2010-2014 only) ---')
print(f'  bii_loss coeff: {fe_short_res.params["bii_loss"]:.4f}  '
      f'(SE: {fe_short_res.std_errors["bii_loss"]:.4f}, '
      f'p: {fe_short_res.pvalues["bii_loss"]:.4f})')

print('\n--- RE (2010-2014 only) ---')
print(f'  bii_loss coeff: {re_short_res.params["bii_loss"]:.4f}  '
      f'(SE: {re_short_res.std_errors["bii_loss"]:.4f}, '
      f'p: {re_short_res.pvalues["bii_loss"]:.4f})')


### 9.2 - Exclude Dominant Exporters (Ireland & Brazil)
Ireland and Brazil account for the vast majority of UK beef imports.
We check whether results are driven by these two countries.


In [ ]:
excl = panel[~panel.reset_index()['iso3'].isin(['IRL', 'BRA']).values].copy()
print(f'Panel excluding IRL & BRA: {excl.shape[0]} obs, '
      f'{excl.reset_index()["iso3"].nunique()} countries')

dep_excl = excl['log_uk_import_qty_t']

fe_excl = PanelOLS(dep_excl, excl[['bii_loss']], entity_effects=True)
fe_excl_res = fe_excl.fit(cov_type='clustered', cluster_entity=True)

print(f'\n  bii_loss coeff: {fe_excl_res.params["bii_loss"]:.4f}  '
      f'(SE: {fe_excl_res.std_errors["bii_loss"]:.4f}, '
      f'p: {fe_excl_res.pvalues["bii_loss"]:.4f})')
print(f'  R-sq (within): {fe_excl_res.rsquared_within:.4f}')


### 9.3 - Balanced Sub-Panel (Countries with >= 10 Years)
Restricts to countries observed in most years to reduce attrition bias.


In [ ]:
year_counts = panel.reset_index().groupby('iso3')['year'].count()
keep_iso3 = year_counts[year_counts >= 10].index

balanced = panel[panel.reset_index()['iso3'].isin(keep_iso3).values].copy()
print(f'Near-balanced panel (>=10 years): {balanced.shape[0]} obs, '
      f'{balanced.reset_index()["iso3"].nunique()} countries')

dep_bal = balanced['log_uk_import_qty_t']

fe_bal = PanelOLS(dep_bal, balanced[['bii_loss']], entity_effects=True)
fe_bal_res = fe_bal.fit(cov_type='clustered', cluster_entity=True)

print(f'\n  bii_loss coeff: {fe_bal_res.params["bii_loss"]:.4f}  '
      f'(SE: {fe_bal_res.std_errors["bii_loss"]:.4f}, '
      f'p: {fe_bal_res.pvalues["bii_loss"]:.4f})')
print(f'  R-sq (within): {fe_bal_res.rsquared_within:.4f}')


---
## 10 - Summary of Results


In [ ]:
results = pd.DataFrame({
    'Model': [
        'Pooled OLS',
        'FE (entity)',
        'FE (entity + time)',
        'Random Effects',
        'FE - 2010-2014 only',
        'FE - excl. IRL & BRA',
        'FE - balanced (>=10 yrs)',
    ],
    'beta_bii_loss': [
        pooled_res.params['bii_loss'],
        fe_res.params['bii_loss'],
        fe2_res.params['bii_loss'],
        re_res.params['bii_loss'],
        fe_short_res.params['bii_loss'],
        fe_excl_res.params['bii_loss'],
        fe_bal_res.params['bii_loss'],
    ],
    'SE': [
        pooled_res.std_errors['bii_loss'],
        fe_res.std_errors['bii_loss'],
        fe2_res.std_errors['bii_loss'],
        re_res.std_errors['bii_loss'],
        fe_short_res.std_errors['bii_loss'],
        fe_excl_res.std_errors['bii_loss'],
        fe_bal_res.std_errors['bii_loss'],
    ],
    'p_value': [
        pooled_res.pvalues['bii_loss'],
        fe_res.pvalues['bii_loss'],
        fe2_res.pvalues['bii_loss'],
        re_res.pvalues['bii_loss'],
        fe_short_res.pvalues['bii_loss'],
        fe_excl_res.pvalues['bii_loss'],
        fe_bal_res.pvalues['bii_loss'],
    ],
    'N': [
        int(pooled_res.nobs),
        int(fe_res.nobs),
        int(fe2_res.nobs),
        int(re_res.nobs),
        int(fe_short_res.nobs),
        int(fe_excl_res.nobs),
        int(fe_bal_res.nobs),
    ],
})

results['Sig.'] = results['p_value'].apply(
    lambda p: '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
)

print('Summary of all panel regression results')
print('=' * 85)
print(results.to_string(index=False, float_format='{:.4f}'.format))


In [ ]:
# Save results table
results.to_csv(os.path.join(CLEAN, 'panel_regression_results.csv'), index=False)
print('Saved => data/clean/panel_regression_results.csv')


---
## 11 - Interpretation

### Key Findings

1. **F-test for entity effects** - if significant, pooled OLS is inappropriate and
   we need to account for unobserved country heterogeneity.

2. **Hausman test** - determines whether Fixed Effects or Random Effects is the
   appropriate estimator.

3. **Coefficient on `bii_loss`** - interpretation depends on the preferred model:
   - *FE*: a one-unit increase in biodiversity loss **within a country over time**
     is associated with a beta-unit change in log UK beef import quantity.
   - *RE*: same, but exploits both within- and between-country variation.

4. **Robustness**: restricting to the 2010-2014 sub-period, excluding dominant
   exporters, and using a near-balanced panel tests whether the main result
   is sensitive to these choices.

### Limitations
- The panel uses **historical BIO data only (2001–2014)** — no model projections.
- The panel is **unbalanced** (not all countries trade in every year).
- **Omitted variable bias** is possible - FE controls for time-invariant
  confounders but not time-varying ones (e.g., trade policy changes).
- With ~32 countries the cluster count is modest, which may affect
  clustered standard error inference.
